# FE OCR v1 — GCS keyframes to schema-ready OCR features

This Kaggle notebook streams AutoShot keyframes from Google Cloud Storage (GCS), detects text with PaddleOCR, recognizes Vietnamese text with VietOCR, and writes resumable artifacts for a later Supabase/PostgreSQL and Zilliz ingestion stage.

The notebook follows four operational sections: **Config**, **Data**, **Model**, and **Run**. Every expensive run is opt-in; a default **Run All** performs only the dry run.

# 1. Config

Edit only the following cell for normal operation. It contains GCS locations, batch selection, run-mode switches, model versions, CPU/GPU tuning, retry behavior, and output controls.

Defaults follow `README(1).MD`: dataset `ai_challenge_2025`, keyframe profile `autoshot_v1`, and batches `L21`–`L30`. To use the earlier datasets, replace `EXPECTED_BATCHES` with `K01`–`K10` or `K11`–`K20`.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple
import os


@dataclass(frozen=True)
class OCRConfig:
    """Central configuration for discovery, download, OCR, and artifact writing."""

    # Safe run-mode switches. Run All executes only the dry run by default.
    RUN_DRY_RUN: bool = True
    RUN_SMOKE_TEST: bool = False
    RUN_DEMO_ONE_BATCH: bool = False
    CONFIRM_FULL_RUN: str = ""  # Set exactly to RUN_FULL_OCR.

    # Dataset and GCS layout from README(1).MD.
    DATASET_ID: str = "ai_challenge_2025"
    KEYFRAME_PROFILE: str = "autoshot_v1"
    OCR_PROFILE: str = "ocr_v1"
    GCS_BUCKET: str = ""  # Prefer Kaggle Secret GCS_BUCKET.
    GCS_BUCKET_SECRET_NAME: str = "GCS_BUCKET"
    GCS_CREDENTIALS_JSON_SECRET_NAME: str = "GCS_CREDENTIALS_JSON"
    GCS_CREDENTIALS_FILE: str = ""
    KEYFRAME_PREFIX: str = "processed/keyframes"
    FEATURE_PREFIX: str = "processed/features"
    EXPECTED_BATCHES: Tuple[str, ...] = tuple(f"L{i:02d}" for i in range(21, 31))

    # Run selections. "all" expands to EXPECTED_BATCHES.
    DRY_RUN_BATCHES: str = "L21"
    SMOKE_BATCHES: str = "L21"
    SMOKE_MAX_FRAMES: int = 8
    DEMO_BATCHES: str = "L21"  # Must resolve to exactly one data batch.
    DEMO_MAX_FRAMES: Optional[int] = 256  # None processes the entire data batch.
    FULL_BATCHES: str = "all"
    FULL_MAX_FRAMES_PER_BATCH: Optional[int] = None

    # Dependency bootstrap. Set False after packages are installed in the session.
    INSTALL_DEPENDENCIES: bool = True
    REMOVE_CPU_PADDLE_BEFORE_INSTALL: bool = True
    PADDLEPADDLE_GPU_SPEC: str = "paddlepaddle-gpu==3.2.2"
    PADDLE_WHEEL_INDEX: str = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"
    PADDLEOCR_SPEC: str = "paddleocr==3.7.0"
    VIETOCR_SPEC: str = "vietocr==0.3.13"

    # OCR models. vgg_seq2seq is faster; vgg_transformer is usually more accurate.
    DETECTION_MODEL: str = "PP-OCRv5_mobile_det"
    RECOGNITION_MODEL: str = "vgg_seq2seq"
    DET_LIMIT_SIDE_LEN: int = 960
    DET_LIMIT_TYPE: str = "max"
    MIN_CROP_HEIGHT: int = 5
    MIN_CROP_WIDTH: int = 5
    CROP_PADDING_PX: int = 12
    LINE_Y_CENTER_SCALE: float = 0.75
    LINE_X_GAP_SCALE: float = 5.0

    # Throughput controls. Reduce OCR batch sizes first if GPU memory is exhausted.
    PIPELINE_BATCH_SIZE: int = 128
    OCR_DET_BATCH_SIZE: int = 32
    OCR_RECOG_BATCH_SIZE: int = 128
    DOWNLOAD_WORKERS: int = min(32, max(8, (os.cpu_count() or 4) * 4))
    DECODE_WORKERS: int = min(16, max(4, os.cpu_count() or 4))
    ARTIFACT_UPLOAD_WORKERS: int = 8
    PREFETCH_BATCHES: int = 2
    CPU_THREADS: int = max(1, os.cpu_count() or 4)

    # Reliability, resume, and storage behavior.
    DOWNLOAD_RETRIES: int = 3
    RETRY_BACKOFF_SECONDS: float = 1.0
    SKIP_EXISTING_LOCAL: bool = True
    FALLBACK_TO_SINGLE_ON_BATCH_ERROR: bool = True
    CLEAN_CACHE_AFTER_BATCH: bool = True
    FLUSH_EVERY_BATCHES: int = 1
    UPLOAD_RUN_ARTIFACTS: bool = True
    UPLOAD_SMOKE_ARTIFACTS: bool = False
    UPLOAD_DRY_RUN_ARTIFACTS: bool = False
    OVERWRITE_GCS_ARTIFACTS: bool = True
    SESSION_TAG: str = ""  # Empty creates one UTC tag when the runtime is initialized.

    # Kaggle-local paths.
    OUTPUT_ROOT: Path = Path("/kaggle/working/feature_extraction_runs")
    CACHE_ROOT: Path = Path("/kaggle/working/fe_ocr_cache")


CFG = OCRConfig()
print(CFG)


## Dependency bootstrap

This cell installs the pinned OCR stack before any PaddlePaddle import. The default PaddlePaddle wheel matches the CUDA 12.6 setup used by the reference notebook. If Kaggle changes its CUDA runtime, update `PADDLE_WHEEL_INDEX` in Config to the matching official wheel channel. Internet must be enabled for first-time installation and model downloads.

In [ ]:
import subprocess
import sys


def run_pip(arguments, check=True):
    """Run pip with the active notebook interpreter and stream its output."""
    command = [sys.executable, "-m", "pip", *arguments]
    print("Running:", " ".join(command))
    return subprocess.run(command, check=check)


if CFG.INSTALL_DEPENDENCIES:
    if CFG.REMOVE_CPU_PADDLE_BEFORE_INSTALL:
        run_pip(["uninstall", "-y", "paddlepaddle"], check=False)
    run_pip([
        "install", "--quiet", "--upgrade", CFG.PADDLEPADDLE_GPU_SPEC,
        "--index-url", CFG.PADDLE_WHEEL_INDEX,
    ])
    run_pip([
        "install", "--quiet", "--upgrade-strategy", "only-if-needed",
        "google-cloud-storage>=2.18,<4",
        CFG.PADDLEOCR_SPEC,
        CFG.VIETOCR_SPEC,
    ])
    print("Dependency installation completed.")
else:
    print("Dependency installation skipped by Config.")


## Runtime and hardware initialization

This cell imports the installed libraries, enables safe CUDA acceleration, assigns CPU threads, creates the session tag used by all run IDs, and prints a compact hardware report.

In [ ]:
import csv
import json
import logging
import math
import re
import shutil
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import paddle
import torch
from PIL import Image

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("FLAGS_use_cuda_managed_memory", "false")

torch.set_num_threads(CFG.CPU_THREADS)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

SESSION_TAG = CFG.SESSION_TAG.strip() or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PADDLE_HAS_CUDA = bool(paddle.is_compiled_with_cuda())

print(f"session_tag={SESSION_TAG}")
print(f"cpu_count={os.cpu_count()} torch_threads={torch.get_num_threads()}")
print(f"torch={torch.__version__} torch_device={TORCH_DEVICE} torch_cuda={torch.version.cuda}")
print(f"paddle={paddle.__version__} paddle_cuda={PADDLE_HAS_CUDA}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)} vram_gb={torch.cuda.get_device_properties(0).total_memory / 2**30:.2f}")


# 2. Data

The Data section authenticates without writing service-account JSON to disk. It accepts Kaggle Secrets or environment variables named `GCS_BUCKET` and `GCS_CREDENTIALS_JSON`, with an optional credential-file fallback from Config.

In [ ]:
from google.cloud import storage
from google.oauth2 import service_account


def get_secret_or_env(name: str) -> str:
    """Return a Kaggle secret when available, otherwise an environment variable."""
    value = os.getenv(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(name) or "").strip()
    except Exception:
        return ""


def split_bucket_and_root(value: str) -> Tuple[str, str]:
    """Split a plain bucket or gs://bucket/optional-root into bucket and root prefix."""
    value = value.strip().rstrip("/")
    if value.startswith("gs://"):
        value = value[5:]
    if not value:
        raise ValueError("GCS bucket is empty. Set the GCS_BUCKET Kaggle Secret or CFG.GCS_BUCKET.")
    parts = value.split("/", 1)
    return parts[0], parts[1].strip("/") if len(parts) == 2 else ""


def join_object_name(*parts: str) -> str:
    """Join GCS object-name components without duplicate separators."""
    return "/".join(str(part).strip("/") for part in parts if str(part).strip("/"))


def parse_gcs_uri(uri: str) -> Tuple[str, str]:
    """Parse gs://bucket/object into its bucket and object name."""
    if not uri.startswith("gs://"):
        raise ValueError(f"Not a GCS URI: {uri}")
    body = uri[5:]
    bucket, separator, object_name = body.partition("/")
    if not bucket or not separator or not object_name:
        raise ValueError(f"Incomplete GCS URI: {uri}")
    return bucket, object_name


def build_gcs_context() -> Tuple[storage.Client, str, str]:
    """Create the authenticated storage client and return client, bucket, and optional root prefix."""
    bucket_value = CFG.GCS_BUCKET.strip() or get_secret_or_env(CFG.GCS_BUCKET_SECRET_NAME)
    bucket_name, root_prefix = split_bucket_and_root(bucket_value)

    credentials_json = get_secret_or_env(CFG.GCS_CREDENTIALS_JSON_SECRET_NAME)
    if credentials_json:
        info = json.loads(credentials_json)
        credentials = service_account.Credentials.from_service_account_info(info)
        client = storage.Client(project=info.get("project_id"), credentials=credentials)
    elif CFG.GCS_CREDENTIALS_FILE:
        credentials = service_account.Credentials.from_service_account_file(CFG.GCS_CREDENTIALS_FILE)
        client = storage.Client(credentials=credentials, project=getattr(credentials, "project_id", None))
    else:
        client = storage.Client()
    return client, bucket_name, root_prefix


GCS_CLIENT, GCS_BUCKET_NAME, GCS_ROOT_PREFIX = build_gcs_context()
GCS_BUCKET_HANDLE = GCS_CLIENT.bucket(GCS_BUCKET_NAME)
print(f"Authenticated bucket=gs://{GCS_BUCKET_NAME} root_prefix={GCS_ROOT_PREFIX or '<none>'}")


## Manifest discovery and schema normalization

The source of truth is each video's `frames_manifest.jsonl`. This preserves the README keys (`dataset_id`, `batch_id`, `video_id`, `shot_id`, `keyframe_id`, frame metadata, and GCS URI). Filename parsing is used only as a defensive fallback for incomplete legacy manifests.

In [ ]:
FRAME_NAME_RE = re.compile(
    r"^shot_(?P<shot_index>\d+)_(?P<frame_type>first|middle|last)_f(?P<frame_idx>\d+)\.(?:jpg|jpeg|png)$",
    re.IGNORECASE,
)


def resolve_batches(selection: Any) -> List[str]:
    """Resolve comma-separated/list batch selection and validate it against EXPECTED_BATCHES."""
    if isinstance(selection, str):
        tokens = [token.strip().upper() for token in selection.split(",") if token.strip()]
    else:
        tokens = [str(token).strip().upper() for token in selection if str(token).strip()]
    if tokens == ["ALL"]:
        tokens = list(CFG.EXPECTED_BATCHES)
    unknown = sorted(set(tokens) - set(CFG.EXPECTED_BATCHES))
    if unknown:
        raise ValueError(f"Unexpected batches {unknown}; edit CFG.EXPECTED_BATCHES if these are intentional.")
    return list(dict.fromkeys(tokens))


def normalize_manifest_record(raw: Dict[str, Any], manifest_blob_name: str, batch_id: str) -> Dict[str, Any]:
    """Normalize one keyframe-manifest row while preserving identifiers from the source contract."""
    parent_prefix = manifest_blob_name.rsplit("/", 1)[0]
    image_uri = str(raw.get("image_gcs_uri") or "").strip()
    image_rel_path = str(raw.get("image_rel_path") or "").strip()
    if not image_uri:
        storage_key = str(raw.get("image_storage_key") or "").strip()
        candidate_name = storage_key or image_rel_path
        if candidate_name.startswith("gs://"):
            image_uri = candidate_name
        elif candidate_name:
            if storage_key:
                object_name = (
                    storage_key
                    if not GCS_ROOT_PREFIX or storage_key.startswith(GCS_ROOT_PREFIX + "/")
                    else join_object_name(GCS_ROOT_PREFIX, storage_key)
                )
            else:
                object_name = join_object_name(parent_prefix, candidate_name)
            image_uri = f"gs://{GCS_BUCKET_NAME}/{object_name}"

    image_name = Path(parse_gcs_uri(image_uri)[1]).name if image_uri else Path(image_rel_path).name
    match = FRAME_NAME_RE.match(image_name)
    video_id = str(raw.get("video_id") or "").strip()
    if not video_id:
        for part in manifest_blob_name.split("/"):
            if part.startswith("video_id="):
                video_id = part.split("=", 1)[1]
                break
    frame_idx = raw.get("frame_idx")
    frame_type = str(raw.get("frame_type") or "").strip().lower()
    shot_index = None
    if match:
        frame_idx = int(frame_idx) if frame_idx is not None else int(match.group("frame_idx"))
        frame_type = frame_type or match.group("frame_type").lower()
        shot_index = int(match.group("shot_index"))

    shot_id = str(raw.get("shot_id") or "").strip()
    if not shot_id and video_id and shot_index is not None:
        shot_id = f"{video_id}_shot_{shot_index:04d}"
    keyframe_id = str(raw.get("keyframe_id") or "").strip()
    if not keyframe_id and video_id and image_name:
        keyframe_id = f"{video_id}_{Path(image_name).stem}"

    if not image_uri or not video_id or not keyframe_id:
        raise ValueError(
            f"Manifest row is missing image_gcs_uri/video_id/keyframe_id in {manifest_blob_name}: {raw}"
        )
    _, image_storage_key = parse_gcs_uri(image_uri)
    return {
        "dataset_id": str(raw.get("dataset_id") or CFG.DATASET_ID),
        "batch_id": str(raw.get("batch_id") or batch_id).upper(),
        "video_id": video_id,
        "shot_id": shot_id,
        "keyframe_id": keyframe_id,
        "frame_idx": int(frame_idx) if frame_idx is not None else None,
        "frame_sec": float(raw["frame_sec"]) if raw.get("frame_sec") is not None else None,
        "frame_type": frame_type,
        "image_rel_path": image_rel_path or image_name,
        "image_storage_key": str(raw.get("image_storage_key") or image_storage_key),
        "image_gcs_uri": image_uri,
        "source_profile": CFG.KEYFRAME_PROFILE,
    }


def read_manifest_blob(blob: storage.Blob, batch_id: str) -> Tuple[str, List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Download and parse one frames_manifest.jsonl object, returning valid rows and parse errors."""
    records, errors = [], []
    text = blob.download_as_text(encoding="utf-8")
    for line_number, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue
        try:
            raw = json.loads(line)
            if raw.get("saved") is False:
                continue
            records.append(normalize_manifest_record(raw, blob.name, batch_id))
        except Exception as exc:
            errors.append({
                "manifest_gcs_uri": f"gs://{GCS_BUCKET_NAME}/{blob.name}",
                "line_number": line_number,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            })
    return blob.name, records, errors


def discover_frame_records(batch_id: str) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], int]:
    """Discover all keyframe rows for one data batch by reading per-video frame manifests in parallel."""
    prefix = join_object_name(
        GCS_ROOT_PREFIX,
        CFG.KEYFRAME_PREFIX,
        f"dataset={CFG.DATASET_ID}",
        f"batch={batch_id}",
        f"profile={CFG.KEYFRAME_PROFILE}",
    ) + "/"
    manifest_blobs = [
        blob for blob in GCS_CLIENT.list_blobs(GCS_BUCKET_NAME, prefix=prefix)
        if blob.name.endswith("/frames_manifest.jsonl")
    ]
    records, errors = [], []
    with ThreadPoolExecutor(max_workers=CFG.DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(read_manifest_blob, blob, batch_id) for blob in manifest_blobs]
        for future in futures:
            try:
                _, rows, row_errors = future.result()
                records.extend(rows)
                errors.extend(row_errors)
            except Exception as exc:
                errors.append({
                    "batch_id": batch_id,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                })

    deduplicated = {}
    for record in records:
        deduplicated[(record["keyframe_id"], record["image_gcs_uri"])] = record
    records = sorted(
        deduplicated.values(),
        key=lambda row: (row["video_id"], row["shot_id"], row["frame_idx"] if row["frame_idx"] is not None else -1),
    )
    return records, errors, len(manifest_blobs)


## Artifact contract and logging

Each run writes an append-safe JSONL file, a flattened CSV, an error stream, the normalized processing manifest, a summary, and a persistent log. The `feature_id` and `(keyframe_id, profile_version)` pair are deterministic join/upsert keys for the later database stage.

In [ ]:
@dataclass(frozen=True)
class RunPaths:
    """Local paths for one mode-and-data-batch run."""

    run_id: str
    run_dir: Path
    artifacts_dir: Path
    log_path: Path
    manifest_path: Path
    jsonl_path: Path
    csv_path: Path
    errors_path: Path
    summary_path: Path
    success_path: Path


CSV_COLUMNS = [
    "feature_id", "feature_type", "dataset_id", "batch_id", "video_id", "shot_id",
    "keyframe_id", "frame_idx", "frame_sec", "frame_type", "image_gcs_uri",
    "ocr_text", "ocr_texts_json", "ocr_regions_json", "text_count", "has_text",
    "ocr_engine", "detection_model", "recognition_model", "profile_version",
    "run_id", "extracted_at", "processing_ms", "status",
]


def make_run_paths(mode: str, batch_id: str) -> RunPaths:
    """Create deterministic local paths for one run within the current notebook session."""
    run_id = f"fe-ocr-{mode.lower().replace('_', '-')}-{batch_id.lower()}-{SESSION_TAG}"
    run_dir = CFG.OUTPUT_ROOT / run_id
    artifacts_dir = run_dir / "artifacts"
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    return RunPaths(
        run_id=run_id,
        run_dir=run_dir,
        artifacts_dir=artifacts_dir,
        log_path=run_dir / "run.log",
        manifest_path=artifacts_dir / "processing_manifest.jsonl",
        jsonl_path=artifacts_dir / "ocr_features.jsonl",
        csv_path=artifacts_dir / "ocr_features.csv",
        errors_path=artifacts_dir / "errors.jsonl",
        summary_path=artifacts_dir / "summary.json",
        success_path=artifacts_dir / "_SUCCESS",
    )


def setup_logger(paths: RunPaths) -> logging.Logger:
    """Build a console-and-file logger without duplicate handlers on cell reruns."""
    logger = logging.getLogger(paths.run_id)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in list(logger.handlers):
        handler.close()
        logger.removeHandler(handler)
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    file_handler = logging.FileHandler(paths.log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger


def write_json(path: Path, payload: Dict[str, Any]) -> None:
    """Atomically write a UTF-8 JSON object."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    """Replace a JSONL file with the supplied rows."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def utc_now_iso() -> str:
    """Return an ISO-8601 UTC timestamp with second precision."""
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def flatten_csv_record(record: Dict[str, Any]) -> Dict[str, Any]:
    """Convert nested OCR arrays to JSON strings for the CSV artifact."""
    row = {key: record.get(key) for key in CSV_COLUMNS}
    row["ocr_texts_json"] = json.dumps(record.get("ocr_texts", []), ensure_ascii=False)
    row["ocr_regions_json"] = json.dumps(record.get("ocr_regions", []), ensure_ascii=False)
    return row


class ArtifactWriter:
    """Append OCR results and errors while supporting local keyframe-level resume."""

    def __init__(self, paths: RunPaths):
        """Open append-safe artifact streams and recover completed local keyframe IDs."""
        self.paths = paths
        self.processed_ids = set()
        if CFG.SKIP_EXISTING_LOCAL and paths.jsonl_path.exists():
            for line in paths.jsonl_path.read_text(encoding="utf-8").splitlines():
                try:
                    row = json.loads(line)
                    if row.get("status") == "ok" and row.get("keyframe_id"):
                        self.processed_ids.add(row["keyframe_id"])
                except Exception:
                    pass

        append = CFG.SKIP_EXISTING_LOCAL and paths.jsonl_path.exists()
        self.jsonl_handle = paths.jsonl_path.open("a" if append else "w", encoding="utf-8")
        csv_exists = append and paths.csv_path.exists() and paths.csv_path.stat().st_size > 0
        self.csv_handle = paths.csv_path.open("a" if append else "w", encoding="utf-8", newline="")
        self.csv_writer = csv.DictWriter(self.csv_handle, fieldnames=CSV_COLUMNS)
        if not csv_exists:
            self.csv_writer.writeheader()
        self.error_handle = paths.errors_path.open("a" if append else "w", encoding="utf-8")

    def write_record(self, record: Dict[str, Any]) -> None:
        """Append one successful OCR record to JSONL and CSV."""
        self.jsonl_handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        self.csv_writer.writerow(flatten_csv_record(record))
        self.processed_ids.add(record["keyframe_id"])

    def write_error(self, error: Dict[str, Any]) -> None:
        """Append one structured frame or manifest error."""
        self.error_handle.write(json.dumps(error, ensure_ascii=False) + "\n")

    def flush(self) -> None:
        """Flush all open artifact streams to disk."""
        self.jsonl_handle.flush()
        self.csv_handle.flush()
        self.error_handle.flush()

    def close(self) -> None:
        """Flush and close every artifact stream."""
        self.flush()
        self.jsonl_handle.close()
        self.csv_handle.close()
        self.error_handle.close()


def upload_run_artifacts(paths: RunPaths, batch_id: str, logger: logging.Logger) -> str:
    """Upload all completed local run artifacts to the configured feature prefix."""
    destination_prefix = join_object_name(
        GCS_ROOT_PREFIX,
        CFG.FEATURE_PREFIX,
        f"dataset={CFG.DATASET_ID}",
        f"batch={batch_id}",
        f"profile={CFG.OCR_PROFILE}",
        f"run_id={paths.run_id}",
    )
    local_files = [path for path in paths.run_dir.rglob("*") if path.is_file()]

    def upload_one(path: Path) -> Tuple[str, int]:
        """Upload one run artifact and return its object name and byte size."""
        relative = path.relative_to(paths.run_dir).as_posix()
        object_name = join_object_name(destination_prefix, relative)
        blob = GCS_BUCKET_HANDLE.blob(object_name)
        if not CFG.OVERWRITE_GCS_ARTIFACTS and blob.exists():
            return object_name, 0
        blob.upload_from_filename(str(path))
        return object_name, path.stat().st_size

    started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=CFG.ARTIFACT_UPLOAD_WORKERS) as pool:
        uploaded = list(pool.map(upload_one, local_files))
    uploaded_bytes = sum(size for _, size in uploaded)
    elapsed = time.perf_counter() - started
    logger.info(
        "[artifact-upload] files=%d bytes=%d seconds=%.2f mbps=%.2f prefix=gs://%s/%s",
        len(uploaded), uploaded_bytes, elapsed,
        uploaded_bytes / 2**20 / max(elapsed, 1e-9), GCS_BUCKET_NAME, destination_prefix,
    )
    return f"gs://{GCS_BUCKET_NAME}/{destination_prefix}/"


# 3. Model

The primary engine mirrors `get-features.ipynb`: PaddleOCR detects text polygons and VietOCR recognizes cropped Vietnamese text lines. Detection is batched on the GPU, image decoding uses CPU threads, and recognition uses `predict_batch` when the installed VietOCR build exposes it; otherwise the engine falls back safely to single-crop prediction.

In [ ]:
def poly_to_xyxy(poly: np.ndarray) -> List[float]:
    """Convert a four-point polygon into an axis-aligned [x1, y1, x2, y2] box."""
    array = np.asarray(poly, dtype=np.float32)
    return [
        float(array[:, 0].min()), float(array[:, 1].min()),
        float(array[:, 0].max()), float(array[:, 1].max()),
    ]


def merge_boxes_by_line(polygons: Sequence[np.ndarray]) -> List[List[int]]:
    """Merge nearby detected words on the same visual line into recognition crops."""
    boxes = sorted((poly_to_xyxy(poly) for poly in polygons), key=lambda box: ((box[1] + box[3]) / 2, box[0]))
    lines: List[Dict[str, Any]] = []
    for box in boxes:
        x1, y1, x2, y2 = box
        height = max(1.0, y2 - y1)
        center_y = (y1 + y2) / 2
        best_line = None
        best_delta = float("inf")
        for line in lines:
            lx1, ly1, lx2, ly2 = line["box"]
            line_height = max(1.0, ly2 - ly1)
            line_center_y = (ly1 + ly2) / 2
            center_delta = abs(center_y - line_center_y)
            horizontal_gap = max(0.0, x1 - lx2)
            same_line = center_delta <= CFG.LINE_Y_CENTER_SCALE * max(height, line_height)
            close_enough = horizontal_gap <= CFG.LINE_X_GAP_SCALE * max(height, line_height)
            if same_line and close_enough and center_delta < best_delta:
                best_line, best_delta = line, center_delta
        if best_line is None:
            lines.append({"box": [x1, y1, x2, y2]})
        else:
            lx1, ly1, lx2, ly2 = best_line["box"]
            best_line["box"] = [min(lx1, x1), min(ly1, y1), max(lx2, x2), max(ly2, y2)]
    return [[int(round(value)) for value in line["box"]] for line in lines]


def crop_xyxy(image_rgb: np.ndarray, box: Sequence[int]) -> Image.Image:
    """Crop a padded, clipped text-line box and return a PIL RGB image."""
    height, width = image_rgb.shape[:2]
    x1, y1, x2, y2 = map(int, box)
    x1 = max(0, x1 - CFG.CROP_PADDING_PX)
    y1 = max(0, y1 - CFG.CROP_PADDING_PX)
    x2 = min(width, x2 + CFG.CROP_PADDING_PX)
    y2 = min(height, y2 + CFG.CROP_PADDING_PX)
    crop = image_rgb[y1:y2, x1:x2]
    if crop.shape[0] < CFG.MIN_CROP_HEIGHT or crop.shape[1] < CFG.MIN_CROP_WIDTH:
        raise ValueError(f"Crop is too small: shape={crop.shape}")
    return Image.fromarray(crop)


def result_to_polygons(result: Any) -> List[np.ndarray]:
    """Extract dt_polys across PaddleOCR Result-object and dictionary variants."""
    payload = result
    json_value = getattr(result, "json", None)
    if json_value is not None:
        try:
            payload = json_value() if callable(json_value) else json_value
        except Exception:
            payload = result
    if isinstance(payload, str):
        try:
            payload = json.loads(payload)
        except Exception:
            payload = {}
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]

    candidates = []
    if isinstance(payload, dict):
        candidates = payload.get("dt_polys", payload.get("polys", []))
    if candidates is None or len(candidates) == 0:
        candidates = getattr(result, "dt_polys", [])
    if candidates is None or len(candidates) == 0:
        try:
            candidates = result["dt_polys"]
        except Exception:
            candidates = []

    polygons = []
    for candidate in candidates:
        polygon = np.asarray(candidate, dtype=np.float32)
        if polygon.shape == (4, 2) and np.isfinite(polygon).all():
            polygons.append(polygon)
    return polygons


def read_rgb_image(path: Path) -> np.ndarray:
    """Decode one image from disk as an RGB NumPy array."""
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"OpenCV could not decode {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


class PaddleVietOCREngine:
    """Batched Paddle text detector plus Vietnamese VietOCR line recognizer."""

    def __init__(self, logger: logging.Logger):
        """Create an unloaded OCR engine bound to the current run logger."""
        self.logger = logger
        self.detector = None
        self.recognizer = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self._batch_recognition_warning_emitted = False

    def load(self) -> "PaddleVietOCREngine":
        """Load detector and recognizer lazily so dry run never allocates GPU memory."""
        from paddleocr import TextDetection
        from vietocr.tool.config import Cfg
        from vietocr.tool.predictor import Predictor

        paddle_device = "gpu:0" if PADDLE_HAS_CUDA else "cpu"
        self.logger.info(
            "Loading OCR models detector=%s recognizer=%s paddle_device=%s torch_device=%s",
            CFG.DETECTION_MODEL, CFG.RECOGNITION_MODEL, paddle_device, self.device,
        )
        self.detector = TextDetection(
            model_name=CFG.DETECTION_MODEL,
            device=paddle_device,
            limit_side_len=CFG.DET_LIMIT_SIDE_LEN,
            limit_type=CFG.DET_LIMIT_TYPE,
        )
        recognition_config = Cfg.load_config_from_name(CFG.RECOGNITION_MODEL)
        recognition_config["cnn"]["pretrained"] = False
        recognition_config["device"] = self.device
        self.recognizer = Predictor(recognition_config)
        self.recognizer.model.eval()
        self.logger.info("OCR models loaded successfully")
        return self

    def _predict_detector(self, paths: Sequence[Path]) -> List[Any]:
        """Run Paddle detection in configurable chunks with a per-image compatibility fallback."""
        outputs: List[Any] = []
        for start in range(0, len(paths), CFG.OCR_DET_BATCH_SIZE):
            chunk = [str(path) for path in paths[start:start + CFG.OCR_DET_BATCH_SIZE]]
            try:
                try:
                    chunk_outputs = list(self.detector.predict(chunk, batch_size=len(chunk)))
                except TypeError:
                    chunk_outputs = list(self.detector.predict(chunk))
                if len(chunk_outputs) != len(chunk):
                    raise RuntimeError(f"Detector returned {len(chunk_outputs)} outputs for {len(chunk)} inputs")
                outputs.extend(chunk_outputs)
            except Exception:
                for path in chunk:
                    outputs.extend(list(self.detector.predict(path)))
        return outputs

    def _predict_recognizer(self, crops: Sequence[Image.Image]) -> Tuple[List[str], List[Any]]:
        """Recognize text crops in batches when supported and return text plus optional confidence."""
        texts, confidences = [], []
        for start in range(0, len(crops), CFG.OCR_RECOG_BATCH_SIZE):
            chunk = list(crops[start:start + CFG.OCR_RECOG_BATCH_SIZE])
            used_batch_api = False
            if hasattr(self.recognizer, "predict_batch"):
                try:
                    with torch.inference_mode():
                        try:
                            prediction = self.recognizer.predict_batch(chunk, return_prob=True)
                        except TypeError:
                            prediction = self.recognizer.predict_batch(chunk)
                    if isinstance(prediction, tuple) and len(prediction) == 2:
                        chunk_texts, chunk_confidences = prediction
                    else:
                        chunk_texts, chunk_confidences = prediction, [None] * len(chunk)
                    if len(chunk_texts) != len(chunk):
                        raise RuntimeError("VietOCR predict_batch returned an unexpected length")
                    texts.extend(str(text).strip() for text in chunk_texts)
                    confidences.extend(
                        float(value) if value is not None and np.isscalar(value) else None
                        for value in chunk_confidences
                    )
                    used_batch_api = True
                except Exception as exc:
                    if not self._batch_recognition_warning_emitted:
                        self.logger.warning("VietOCR batch API failed; using single-crop fallback: %s", exc)
                        self._batch_recognition_warning_emitted = True
            if not used_batch_api:
                for crop in chunk:
                    with torch.inference_mode():
                        try:
                            prediction = self.recognizer.predict(crop, return_prob=True)
                        except TypeError:
                            prediction = self.recognizer.predict(crop)
                    if isinstance(prediction, tuple) and len(prediction) == 2:
                        text, confidence = prediction
                    else:
                        text, confidence = prediction, None
                    texts.append(str(text).strip())
                    confidences.append(float(confidence) if confidence is not None and np.isscalar(confidence) else None)
        return texts, confidences

    def infer(self, image_paths: Sequence[Path]) -> List[Dict[str, Any]]:
        """Detect, crop, recognize, and return ordered OCR regions for every input image."""
        detection_results = self._predict_detector(image_paths)
        with ThreadPoolExecutor(max_workers=CFG.DECODE_WORKERS) as pool:
            decoded = list(pool.map(read_rgb_image, image_paths))

        crops: List[Image.Image] = []
        crop_owners: List[int] = []
        crop_boxes: List[List[int]] = []
        for image_index, (image_rgb, detection_result) in enumerate(zip(decoded, detection_results)):
            for box in merge_boxes_by_line(result_to_polygons(detection_result)):
                try:
                    crops.append(crop_xyxy(image_rgb, box))
                    crop_owners.append(image_index)
                    crop_boxes.append(box)
                except ValueError:
                    continue

        texts, confidences = self._predict_recognizer(crops) if crops else ([], [])
        outputs = [{"ocr_texts": [], "ocr_regions": []} for _ in image_paths]
        for owner, box, text, confidence in zip(crop_owners, crop_boxes, texts, confidences):
            clean_text = " ".join(str(text).split())
            if not clean_text:
                continue
            region = {"text": clean_text, "bbox_xyxy": box, "confidence": confidence}
            outputs[owner]["ocr_regions"].append(region)
        for output in outputs:
            output["ocr_regions"].sort(key=lambda region: (region["bbox_xyxy"][1], region["bbox_xyxy"][0]))
            output["ocr_texts"] = [region["text"] for region in output["ocr_regions"]]
        return outputs


OCR_ENGINE_CACHE: Dict[str, PaddleVietOCREngine] = {}


def get_ocr_engine(logger: logging.Logger) -> PaddleVietOCREngine:
    """Return the session-level OCR model cache, loading weights on first use only."""
    cache_key = f"{CFG.DETECTION_MODEL}:{CFG.RECOGNITION_MODEL}"
    if cache_key not in OCR_ENGINE_CACHE:
        OCR_ENGINE_CACHE[cache_key] = PaddleVietOCREngine(logger).load()
    else:
        OCR_ENGINE_CACHE[cache_key].logger = logger
    return OCR_ENGINE_CACHE[cache_key]


## Streaming execution engine

This cell overlaps GCS downloads with GPU work through a bounded prefetch queue. It logs every download and OCR mini-batch with counts, completion percentage, elapsed time, throughput, and ETA. Failed batch inference is recursively split to isolate bad frames; successful results remain append-safe.

In [ ]:
def local_cache_path(record: Dict[str, Any]) -> Path:
    """Map a keyframe record to a collision-resistant Kaggle cache path."""
    _, object_name = parse_gcs_uri(record["image_gcs_uri"])
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(object_name).name)
    return CFG.CACHE_ROOT / record["batch_id"] / record["video_id"] / safe_name


def download_one_frame(record: Dict[str, Any]) -> Dict[str, Any]:
    """Download one GCS image with retry and return structured transfer metrics."""
    local_path = local_cache_path(record)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    if local_path.exists() and local_path.stat().st_size > 0:
        return {
            "record": record, "local_path": local_path, "bytes": local_path.stat().st_size,
            "cached": True, "seconds": time.perf_counter() - started, "error": None,
        }
    bucket_name, object_name = parse_gcs_uri(record["image_gcs_uri"])
    last_error = None
    for attempt in range(1, CFG.DOWNLOAD_RETRIES + 1):
        try:
            GCS_CLIENT.bucket(bucket_name).blob(object_name).download_to_filename(str(local_path))
            return {
                "record": record, "local_path": local_path, "bytes": local_path.stat().st_size,
                "cached": False, "seconds": time.perf_counter() - started, "error": None,
            }
        except Exception as exc:
            last_error = exc
            if local_path.exists():
                local_path.unlink()
            if attempt < CFG.DOWNLOAD_RETRIES:
                time.sleep(CFG.RETRY_BACKOFF_SECONDS * (2 ** (attempt - 1)))
    return {
        "record": record, "local_path": local_path, "bytes": 0, "cached": False,
        "seconds": time.perf_counter() - started,
        "error": {"error_type": type(last_error).__name__, "error_message": str(last_error)},
    }


def iter_prefetched_download_batches(
    records: Sequence[Dict[str, Any]], logger: logging.Logger
) -> Iterator[Tuple[int, List[Dict[str, Any]]]]:
    """Yield downloaded mini-batches while prefetching the next batch in a shared thread pool."""
    chunks = [
        list(records[start:start + CFG.PIPELINE_BATCH_SIZE])
        for start in range(0, len(records), CFG.PIPELINE_BATCH_SIZE)
    ]
    total_batches = len(chunks)
    total_frames = len(records)
    pending = deque()
    run_started = time.perf_counter()

    def resolve_pending(item):
        """Resolve one queued transfer batch and emit its progress log."""
        batch_index, batch_started, futures = item
        results = [future.result() for future in futures]
        elapsed = time.perf_counter() - batch_started
        completed = min((batch_index + 1) * CFG.PIPELINE_BATCH_SIZE, total_frames)
        bytes_downloaded = sum(result["bytes"] for result in results if not result["cached"])
        cached = sum(bool(result["cached"]) for result in results)
        failed = sum(result["error"] is not None for result in results)
        logger.info(
            "[download %d/%d %.1f%%] frames=%d completed=%d/%d cached=%d failed=%d "
            "batch_s=%.2f run_s=%.2f frames_s=%.2f MB_s=%.2f",
            batch_index + 1, total_batches, 100.0 * completed / max(total_frames, 1),
            len(results), completed, total_frames, cached, failed, elapsed,
            time.perf_counter() - run_started, len(results) / max(elapsed, 1e-9),
            bytes_downloaded / 2**20 / max(elapsed, 1e-9),
        )
        return batch_index, results

    with ThreadPoolExecutor(max_workers=CFG.DOWNLOAD_WORKERS) as pool:
        for batch_index, chunk in enumerate(chunks):
            batch_started = time.perf_counter()
            futures = [pool.submit(download_one_frame, record) for record in chunk]
            pending.append((batch_index, batch_started, futures))
            if len(pending) >= max(1, CFG.PREFETCH_BATCHES):
                yield resolve_pending(pending.popleft())
        while pending:
            yield resolve_pending(pending.popleft())


def infer_with_recovery(
    engine: PaddleVietOCREngine,
    items: Sequence[Tuple[Dict[str, Any], Path]],
) -> List[Tuple[Dict[str, Any], Path, Any, Any]]:
    """Infer a batch and recursively split failures so one bad frame does not discard good frames."""
    if not items:
        return []
    records = [item[0] for item in items]
    paths = [item[1] for item in items]
    try:
        predictions = engine.infer(paths)
        if len(predictions) != len(items):
            raise RuntimeError(f"OCR returned {len(predictions)} predictions for {len(items)} frames")
        return [(record, path, prediction, None) for record, path, prediction in zip(records, paths, predictions)]
    except Exception as exc:
        message = str(exc).lower()
        if "out of memory" in message and torch.cuda.is_available():
            torch.cuda.empty_cache()
        if CFG.FALLBACK_TO_SINGLE_ON_BATCH_ERROR and len(items) > 1:
            midpoint = len(items) // 2
            return infer_with_recovery(engine, items[:midpoint]) + infer_with_recovery(engine, items[midpoint:])
        return [(record, path, None, exc) for record, path in items]


def build_ocr_record(
    source: Dict[str, Any], prediction: Dict[str, Any], run_id: str, processing_ms: float
) -> Dict[str, Any]:
    """Build one schema-ready OCR feature record linked to the README keyframe contract."""
    texts = prediction.get("ocr_texts", [])
    return {
        "feature_id": f"{source['keyframe_id']}:{CFG.OCR_PROFILE}",
        "feature_type": "ocr",
        "dataset_id": source["dataset_id"],
        "batch_id": source["batch_id"],
        "video_id": source["video_id"],
        "shot_id": source["shot_id"],
        "keyframe_id": source["keyframe_id"],
        "frame_idx": source["frame_idx"],
        "frame_sec": source["frame_sec"],
        "frame_type": source["frame_type"],
        "image_gcs_uri": source["image_gcs_uri"],
        "ocr_text": "\n".join(texts),
        "ocr_texts": texts,
        "ocr_regions": prediction.get("ocr_regions", []),
        "text_count": len(texts),
        "has_text": bool(texts),
        "ocr_engine": "paddle_detection+vietocr_recognition",
        "detection_model": CFG.DETECTION_MODEL,
        "recognition_model": CFG.RECOGNITION_MODEL,
        "profile_version": CFG.OCR_PROFILE,
        "run_id": run_id,
        "extracted_at": utc_now_iso(),
        "processing_ms": round(float(processing_ms), 3),
        "status": "ok",
    }


def create_error(source: Dict[str, Any], stage: str, exc: Any, run_id: str) -> Dict[str, Any]:
    """Create a consistent structured error record for retry and auditing."""
    if isinstance(exc, dict):
        error_type = exc.get("error_type", "Error")
        error_message = exc.get("error_message", str(exc))
    else:
        error_type = type(exc).__name__
        error_message = str(exc)
    return {
        "run_id": run_id,
        "stage": stage,
        "dataset_id": source.get("dataset_id"),
        "batch_id": source.get("batch_id"),
        "video_id": source.get("video_id"),
        "shot_id": source.get("shot_id"),
        "keyframe_id": source.get("keyframe_id"),
        "image_gcs_uri": source.get("image_gcs_uri"),
        "error_type": error_type,
        "error_message": error_message,
        "failed_at": utc_now_iso(),
    }


def execute_batch_run(
    mode: str,
    batch_id: str,
    max_frames: Any,
    load_model: bool,
    upload_artifacts: bool,
) -> Dict[str, Any]:
    """Execute discovery plus optional OCR for one data batch and persist a complete run summary."""
    paths = make_run_paths(mode, batch_id)
    logger = setup_logger(paths)
    run_started = time.perf_counter()
    logger.info("run_id=%s mode=%s batch=%s", paths.run_id, mode, batch_id)

    records, discovery_errors, manifest_count = discover_frame_records(batch_id)
    if max_frames is not None:
        records = records[: int(max_frames)]
    write_jsonl(paths.manifest_path, records)
    logger.info(
        "[discover 100.0%%] batch=%s manifests=%d frames=%d errors=%d",
        batch_id, manifest_count, len(records), len(discovery_errors),
    )

    writer = ArtifactWriter(paths)
    for error in discovery_errors:
        writer.write_error({"run_id": paths.run_id, "stage": "manifest", **error})

    summary = {
        "run_id": paths.run_id,
        "mode": mode,
        "dataset_id": CFG.DATASET_ID,
        "batch_id": batch_id,
        "source_profile": CFG.KEYFRAME_PROFILE,
        "profile_version": CFG.OCR_PROFILE,
        "manifest_count": manifest_count,
        "planned_frames": len(records),
        "skipped_existing_frames": 0,
        "succeeded_frames": 0,
        "failed_frames": len(discovery_errors),
        "frames_with_text": 0,
        "text_lines": 0,
        "started_at": utc_now_iso(),
        "completed_at": None,
        "duration_ms": None,
        "frames_per_min": 0.0,
        "uploaded_gcs_prefix": None,
        "dry_run": not load_model,
        "success": False,
    }

    if not load_model:
        writer.close()
        summary["completed_at"] = utc_now_iso()
        summary["duration_ms"] = round((time.perf_counter() - run_started) * 1000, 3)
        summary["success"] = len(discovery_errors) == 0 and len(records) > 0
        write_json(paths.summary_path, summary)
        if summary["success"]:
            paths.success_path.write_text("", encoding="utf-8")
        if upload_artifacts:
            destination_prefix = join_object_name(
                GCS_ROOT_PREFIX, CFG.FEATURE_PREFIX, f"dataset={CFG.DATASET_ID}",
                f"batch={batch_id}", f"profile={CFG.OCR_PROFILE}", f"run_id={paths.run_id}",
            )
            summary["uploaded_gcs_prefix"] = f"gs://{GCS_BUCKET_NAME}/{destination_prefix}/"
            write_json(paths.summary_path, summary)
            upload_run_artifacts(paths, batch_id, logger)
        logger.info("dry-run complete summary=%s", paths.summary_path)
        return summary

    engine = get_ocr_engine(logger)
    remaining = [record for record in records if record["keyframe_id"] not in writer.processed_ids]
    summary["skipped_existing_frames"] = len(records) - len(remaining)
    total_batches = math.ceil(len(remaining) / CFG.PIPELINE_BATCH_SIZE) if remaining else 0
    attempted = 0

    try:
        for batch_index, download_results in iter_prefetched_download_batches(remaining, logger):
            batch_started = time.perf_counter()
            ready_items = []
            for result in download_results:
                if result["error"] is not None:
                    writer.write_error(create_error(result["record"], "download", result["error"], paths.run_id))
                    summary["failed_frames"] += 1
                else:
                    ready_items.append((result["record"], result["local_path"]))

            inference_started = time.perf_counter()
            inference_results = infer_with_recovery(engine, ready_items)
            inference_elapsed = time.perf_counter() - inference_started
            per_frame_ms = 1000.0 * inference_elapsed / max(len(ready_items), 1)
            for source, local_path, prediction, error in inference_results:
                if error is not None:
                    writer.write_error(create_error(source, "ocr", error, paths.run_id))
                    summary["failed_frames"] += 1
                else:
                    record = build_ocr_record(source, prediction, paths.run_id, per_frame_ms)
                    writer.write_record(record)
                    summary["succeeded_frames"] += 1
                    summary["frames_with_text"] += int(record["has_text"])
                    summary["text_lines"] += record["text_count"]
                if CFG.CLEAN_CACHE_AFTER_BATCH and local_path.exists():
                    local_path.unlink()

            attempted += len(download_results)
            elapsed_run = time.perf_counter() - run_started
            completed_percent = 100.0 * attempted / max(len(remaining), 1)
            throughput = summary["succeeded_frames"] / max(elapsed_run, 1e-9)
            remaining_count = max(0, len(remaining) - attempted)
            eta_seconds = remaining_count / max(throughput, 1e-9)
            logger.info(
                "[ocr %d/%d %.1f%%] frames=%d completed=%d/%d ok=%d failed=%d with_text=%d "
                "batch_s=%.2f infer_s=%.2f run_s=%.2f frames_s=%.2f eta_s=%.1f",
                batch_index + 1, total_batches, completed_percent, len(download_results), attempted,
                len(remaining), summary["succeeded_frames"], summary["failed_frames"],
                summary["frames_with_text"], time.perf_counter() - batch_started, inference_elapsed,
                elapsed_run, throughput, eta_seconds,
            )
            if (batch_index + 1) % max(1, CFG.FLUSH_EVERY_BATCHES) == 0:
                writer.flush()
    finally:
        writer.close()

    duration = time.perf_counter() - run_started
    summary["completed_at"] = utc_now_iso()
    summary["duration_ms"] = round(duration * 1000, 3)
    summary["frames_per_min"] = round(summary["succeeded_frames"] / max(duration, 1e-9) * 60, 3)
    summary["success"] = summary["failed_frames"] == 0 and summary["succeeded_frames"] + summary["skipped_existing_frames"] == len(records)
    if upload_artifacts:
        destination_prefix = join_object_name(
            GCS_ROOT_PREFIX, CFG.FEATURE_PREFIX, f"dataset={CFG.DATASET_ID}",
            f"batch={batch_id}", f"profile={CFG.OCR_PROFILE}", f"run_id={paths.run_id}",
        )
        summary["uploaded_gcs_prefix"] = f"gs://{GCS_BUCKET_NAME}/{destination_prefix}/"
    write_json(paths.summary_path, summary)
    if summary["success"]:
        paths.success_path.write_text("", encoding="utf-8")
    if upload_artifacts:
        upload_run_artifacts(paths, batch_id, logger)
    logger.info(
        "run complete success=%s ok=%d failed=%d duration_s=%.2f frames_per_min=%.2f summary=%s",
        summary["success"], summary["succeeded_frames"], summary["failed_frames"],
        duration, summary["frames_per_min"], paths.summary_path,
    )
    return summary


def run_mode(
    mode: str,
    batch_selection: Any,
    max_frames_per_batch: Any,
    load_model: bool,
    upload_artifacts: bool,
) -> List[Dict[str, Any]]:
    """Run one named mode across its selected data batches and return summaries."""
    batches = resolve_batches(batch_selection)
    if not batches:
        raise ValueError(f"No batches selected for mode={mode}")
    summaries = []
    for batch_id in batches:
        summaries.append(execute_batch_run(
            mode=mode,
            batch_id=batch_id,
            max_frames=max_frames_per_batch,
            load_model=load_model,
            upload_artifacts=upload_artifacts,
        ))
    return summaries


# 4. Run

## Dry run

The dry run authenticates, lists per-video manifests, normalizes the keyframe schema, and writes a local processing manifest. It does **not** load OCR models or download frame images. This is the only mode enabled by default.

In [ ]:
DRY_RUN_RESULTS = []
if CFG.RUN_DRY_RUN:
    DRY_RUN_RESULTS = run_mode(
        mode="dry_run",
        batch_selection=CFG.DRY_RUN_BATCHES,
        max_frames_per_batch=None,
        load_model=False,
        upload_artifacts=CFG.UPLOAD_DRY_RUN_ARTIFACTS,
    )
    display(pd.DataFrame(DRY_RUN_RESULTS))
else:
    print("Dry run disabled. Set CFG.RUN_DRY_RUN=True to enable it.")


## Smoke test

Enable `RUN_SMOKE_TEST` in Config to load both models and process a very small deterministic sample. The default is eight frames from `L21`; artifacts stay local unless `UPLOAD_SMOKE_ARTIFACTS=True`.

In [ ]:
SMOKE_RESULTS = []
if CFG.RUN_SMOKE_TEST:
    SMOKE_RESULTS = run_mode(
        mode="smoke_test",
        batch_selection=CFG.SMOKE_BATCHES,
        max_frames_per_batch=CFG.SMOKE_MAX_FRAMES,
        load_model=True,
        upload_artifacts=CFG.UPLOAD_SMOKE_ARTIFACTS,
    )
    display(pd.DataFrame(SMOKE_RESULTS))
else:
    print("Smoke test disabled. Set CFG.RUN_SMOKE_TEST=True after a successful dry run.")


## Demo: one data batch

Enable `RUN_DEMO_ONE_BATCH` to exercise production behavior for exactly one batch. The default processes 256 frames from `L21`; set `DEMO_MAX_FRAMES=None` to process the whole batch. Completed artifacts are uploaded when `UPLOAD_RUN_ARTIFACTS=True`.

In [ ]:
DEMO_RESULTS = []
if CFG.RUN_DEMO_ONE_BATCH:
    demo_batches = resolve_batches(CFG.DEMO_BATCHES)
    if len(demo_batches) != 1:
        raise ValueError(f"Demo must select exactly one data batch; got {demo_batches}")
    DEMO_RESULTS = run_mode(
        mode="demo",
        batch_selection=demo_batches,
        max_frames_per_batch=CFG.DEMO_MAX_FRAMES,
        load_model=True,
        upload_artifacts=CFG.UPLOAD_RUN_ARTIFACTS,
    )
    display(pd.DataFrame(DEMO_RESULTS))
else:
    print("Demo disabled. Set CFG.RUN_DEMO_ONE_BATCH=True after the smoke test succeeds.")


## Full extraction

This cell is guarded against accidental execution. Set `CONFIRM_FULL_RUN="RUN_FULL_OCR"`, choose `FULL_BATCHES`, and keep `FULL_MAX_FRAMES_PER_BATCH=None` for a complete run. For multiple Kaggle notebooks, assign disjoint batches such as `L21`, `L22`, and so on.

In [ ]:
FULL_RESULTS = []
if CFG.CONFIRM_FULL_RUN == "RUN_FULL_OCR":
    FULL_RESULTS = run_mode(
        mode="full",
        batch_selection=CFG.FULL_BATCHES,
        max_frames_per_batch=CFG.FULL_MAX_FRAMES_PER_BATCH,
        load_model=True,
        upload_artifacts=CFG.UPLOAD_RUN_ARTIFACTS,
    )
    display(pd.DataFrame(FULL_RESULTS))
else:
    print('Full run blocked. Set CFG.CONFIRM_FULL_RUN="RUN_FULL_OCR" only after dry run, smoke test, and demo pass.')


## Inspect local outputs

Use this final cell to list run summaries and preview the newest successful OCR records without loading an entire large JSONL file into memory.

In [ ]:
summary_files = sorted(CFG.OUTPUT_ROOT.glob("*/artifacts/summary.json"))
print(f"Found {len(summary_files)} local run summaries under {CFG.OUTPUT_ROOT}")
for path in summary_files[-10:]:
    summary = json.loads(path.read_text(encoding="utf-8"))
    print(
        path.parent.parent.name,
        f"mode={summary.get('mode')}",
        f"batch={summary.get('batch_id')}",
        f"ok={summary.get('succeeded_frames')}",
        f"failed={summary.get('failed_frames')}",
        f"frames_per_min={summary.get('frames_per_min')}",
    )

feature_files = sorted(CFG.OUTPUT_ROOT.glob("*/artifacts/ocr_features.jsonl"))
if feature_files:
    newest = feature_files[-1]
    preview = []
    with newest.open("r", encoding="utf-8") as handle:
        for _ in range(5):
            line = handle.readline()
            if not line:
                break
            preview.append(json.loads(line))
    print(f"Preview from {newest}")
    display(pd.DataFrame(preview))
